# exp_dani.ipynb — Notebook personal: **Dani**

Notebook personal para el hackathon de predicción de retornos.  
Cada miembro trabaja en su copia; resultados se comparten verbalmente / por chat.  
El mejor enfoque se porta a `COMPETICION.ipynb` para el entregable final.

**Flujo**: experimentar aquí → exportar `models/dani.keras` + `results/dani.json`  
→ `COMPETICION.ipynb` carga y combina los 3 modelos del equipo.

---
⚠️ **GPU workaround obligatorio** (RTX 5070 Ti Blackwell incompatible con TF GPU):  
ejecutar la celda siguiente **antes** de cualquier import de TF/Keras.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # CPU-only — ANTES de cualquier import TF

import sys
sys.path.insert(0, os.getcwd())
import numpy as np
import pandas as pd
import json
import warnings
warnings.simplefilter('ignore')

from utils import (
    create_time_series_data, make_splits, load_data,
    build_model, train_model, train_ensemble,
    evaluate, eval_mae_naive, build_results_df, plot_history,
    FILEPATH_SHARED, V_IN_SHARED, V_OUT_SHARED,
    FFD_D_SHARED, PRICE_COLS_SHARED, RETURN_COLS_SHARED,
)
print('OK — modo CPU activo')

## ☑️ Checklist del enunciado — RELLENAR A MANO antes de ejecutar nada

**Leer el enunciado completo y contestar antes de tocar el código.**

### Qué hay que predecir
- [ ] ¿Regresión (valor continuo) o clasificación (categoría)?  → `TIPO_PROBLEMA = ___`
- [ ] ¿Cuántos días de horizonte de predicción?               → actualizar `V_OUT_SHARED` en `utils.py`
- [ ] ¿Cuántos días de historia de entrada (si lo dice)?       → actualizar `V_IN_SHARED` en `utils.py`
- [ ] ¿Predecir un activo o varios?                           → univariante / multivariante

### Métrica y entregable
- [ ] ¿Qué métrica evalúa el hackathon?   → `METRICA = ___`  (MAE / RMSE / accuracy / otra)
- [ ] ¿Qué hay que entregar exactamente?  → CSV / MAE en test / notebook / otro: ___
- [ ] ¿Hay columna target explícita o hay que derivarla?

### Datos
- [ ] ¿El CSV trae precios o retornos?    → actualizar `PRICE_COLS_SHARED` / `RETURN_COLS_SHARED` en `utils.py`
- [ ] ¿Hay columnas extra (indicadores, covariables)?  → puede necesitar modelo funcional
- [ ] ¿Índice temporal?                  → `shuffle=False` obligatorio si lo es

> ⚠️ `V_IN`, `V_OUT`, `FFD_D` vienen de `utils.py` y son **comunes a todo el equipo**.
> Solo se cambian en `utils.py` con consenso. En este notebook solo ajustar parámetros de modelo.

## 0. Diagnóstico automático del dataset

In [ ]:
FILEPATH_RAW = FILEPATH_SHARED

try:
    ext = os.path.splitext(FILEPATH_RAW)[1].lower()
    df_raw = (pd.read_parquet(FILEPATH_RAW) if ext == '.parquet'
              else pd.read_csv(FILEPATH_RAW, index_col=0, parse_dates=True))

    SEP = '=' * 65
    print(SEP)
    print(f'  DATASET: {FILEPATH_RAW}')
    print(f'  {df_raw.shape[0]:,} filas  x  {df_raw.shape[1]} columnas')
    print(SEP)

    print('\nÍNDICE')
    es_temporal = pd.api.types.is_datetime64_any_dtype(df_raw.index)
    shuffle_msg = '-> shuffle=False obligatorio' if es_temporal else '-> verificar si es serie temporal'
    print(f'  tipo     : {type(df_raw.index).__name__}')
    print(f'  temporal : {es_temporal}  {shuffle_msg}')
    print(f'  primero  : {df_raw.index[0]}')
    print(f'  ultimo   : {df_raw.index[-1]}')

    print('\nCOLUMNAS  (dtype | NaN | n_unicos | [min, max] | clasif?)')
    for col in df_raw.columns:
        ser   = df_raw[col]
        nuniq = ser.nunique()
        nnan  = ser.isna().sum()
        if pd.api.types.is_numeric_dtype(ser):
            rng = f'[{ser.min():.4g}, {ser.max():.4g}]'
            tag = '<- CLASIF?' if nuniq < 20 else ''
        else:
            rng = f'categ ({nuniq} vals)'
            tag = '<- TEXT/CATEG'
        print(f'  {col:28s}  {str(ser.dtype):10s}  NaN={nnan:5d}  '
              f'nuniq={nuniq:6d}  {rng:22s}  {tag}')

    nan_total = df_raw.isna().sum().sum()
    print(f'\nNaN totales: {nan_total}',
          '<- requiere tratamiento' if nan_total > 0 else '<- OK')
    print('\n-> Revisar columnas antes de decidir PRICE_COLS / RETURN_COLS / TARGET')
    print('-> Si nuniq < 20 en la columna target: posiblemente clasificacion, no regresion')
    display(df_raw.head(4))

except FileNotFoundError:
    print(f'Fichero no encontrado: {FILEPATH_RAW}')
    print('   Ajusta FILEPATH_SHARED en utils.py.')

## 1. Configuración

Las **constantes de datos** vienen de `utils.py` (acordadas en equipo — **no cambiar aquí**).  
Los **parámetros de modelo** son personales: experimenta libremente con arquitectura y épocas.

In [ ]:
# ── CONSTANTES DE DATOS — de utils.py, acordadas en equipo, NO modificar ─────
FILEPATH = FILEPATH_SHARED
V_IN     = V_IN_SHARED
V_OUT    = V_OUT_SHARED
FFD_D    = FFD_D_SHARED

# ── PARÁMETROS PERSONALES — experimentar aquí libremente ─────────────────────
EPOCHS_SCREEN = 40    # cribado rápido: 30-50 épocas para descartar modelos
EPOCHS_FULL   = 200   # entreno completo de candidatos
N_CANDIDATOS  = 2     # cuántos modelos pasan del cribado al entreno completo
N_SEEDS       = 5     # semillas para el ensemble final

X_src, y_src = load_data(FILEPATH, price_cols=PRICE_COLS_SHARED,
                          return_cols=RETURN_COLS_SHARED, ffd_d=FFD_D)
display(X_src.head(3))

## 2. Ventanas deslizantes + split temporal

In [ ]:
X, _  = create_time_series_data(X_src, V_IN, V_OUT)
_, y  = create_time_series_data(y_src, V_IN, V_OUT)
X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X, y)

N_FEAT    = X.shape[2]
N_TARGETS = y.shape[1]

print(f'X: {X.shape}   y: {y.shape}')
print(f'Train: {X_tr.shape[0]:,}  Val: {X_v.shape[0]:,}  Test: {X_ts.shape[0]:,}')
print(f'n_features={N_FEAT}  n_targets={N_TARGETS}')

naive_mae = eval_mae_naive(X_ts, y_ts)
print(f'\nBaseline naive (ultimo valor conocido): MAE = {naive_mae:.4f}')
print('-> Este es el suelo minimo a superar en este dataset.')
print('-> Los numeros de la tarea previa (naive~0.018, NN~0.012) pueden NO aplicar aqui.')

## 3. Cribado rápido — descartar los que no aprenden

Entrena las 4 arquitecturas a `EPOCHS_SCREEN` épocas (30–50).  
Objetivo: ver cuáles convergen **en este problema** y descartar las que no aprenden nada.  
Solo las prometedoras pasan al entrenamiento completo.

> Los LR (`LR_MAP`) son el punto de partida de la tarea previa con 23 activos SP500.  
> Si las curvas del cribado son **erráticas o completamente planas**, ajustar LR antes del  
> entreno completo (probar ×10 o ÷10).

In [ ]:
LR_MAP = {'dense': 1e-4, 'lstm': 3e-4, 'cnn1d': 3e-4, 'cnn_lstm': 3e-4}

results_screen = {}
for tipo in ['dense', 'lstm', 'cnn1d', 'cnn_lstm']:
    print(f'\n-- {tipo.upper()} ({EPOCHS_SCREEN} epocas) --')
    model = build_model(tipo, V_IN, n_features=N_FEAT,
                        n_targets=N_TARGETS, lr=LR_MAP[tipo])
    hist  = train_model(model, X_tr, y_tr, X_v, y_v, epochs=EPOCHS_SCREEN)
    res   = evaluate(model, X_tr, X_v, X_ts, y_tr, y_v, y_ts)
    results_screen[tipo] = res
    print(f'  val={res["val"]:.4f}  test={res["test"]:.4f}  dir={res["dir"]:.2%}')
    plot_history(hist, title=f'{tipo} — cribado {EPOCHS_SCREEN} epocas')

df_screen = (pd.DataFrame(results_screen)
               .T[['train', 'val', 'test', 'dir', 'params']]
               .sort_values('val'))
print(f'\n-- RANKING CRIBADO (por val MAE) --')
print(df_screen.to_string())
print(f'\nNaive baseline: {naive_mae:.4f}')

CANDIDATOS = list(df_screen.index[:N_CANDIDATOS])
print(f'\n-> Candidatos para entreno completo: {CANDIDATOS}')
print(f'   Modifica CANDIDATOS = [...] en la celda siguiente si prefieres otros')

In [ ]:
# Ajuste manual (descomenta si quieres cambiar la seleccion del cribado):
# CANDIDATOS = ['lstm', 'cnn1d']

## 4. Entrenamiento completo — solo los candidatos del cribado

In [ ]:
results_full = {}
models_full  = {}   # guardar referencias para exportar en seccion 7

for tipo in CANDIDATOS:
    print(f'\n== {tipo.upper()} — {EPOCHS_FULL} epocas ==')
    model = build_model(tipo, V_IN, n_features=N_FEAT,
                        n_targets=N_TARGETS, lr=LR_MAP[tipo])
    hist  = train_model(model, X_tr, y_tr, X_v, y_v, epochs=EPOCHS_FULL)
    res   = evaluate(model, X_tr, X_v, X_ts, y_tr, y_v, y_ts)
    results_full[tipo] = res
    models_full[tipo]  = model
    print(f'  val={res["val"]:.4f}  test={res["test"]:.4f}  dir={res["dir"]:.2%}')
    plot_history(hist, title=f'{tipo} — {EPOCHS_FULL} epocas')

mejor_tipo   = min(results_full, key=lambda t: results_full[t]['val'])
mejor_modelo = models_full[mejor_tipo]
print(f'\n-> Ganador: {mejor_tipo}  '
      f'val={results_full[mejor_tipo]["val"]:.4f}  '
      f'test={results_full[mejor_tipo]["test"]:.4f}')

## 5. Ensemble del ganador — palanca principal

Promediar `N_SEEDS` semillas del mismo modelo elimina el ruido de inicialización  
y reduce la varianza del MAE. Es la principal palanca de reducción de varianza.

In [ ]:
TIPO_ENSEMBLE = mejor_tipo   # o forzar: TIPO_ENSEMBLE = 'lstm'

print(f'Ensemble: {N_SEEDS} semillas de {TIPO_ENSEMBLE} x {EPOCHS_FULL} epocas\n')
ens = train_ensemble(
    TIPO_ENSEMBLE, X_tr, y_tr, X_v, y_v, X_ts, y_ts,
    V_in=V_IN, n_features=N_FEAT, n_targets=N_TARGETS,
    n_seeds=N_SEEDS, epochs=EPOCHS_FULL, lr=LR_MAP[TIPO_ENSEMBLE],
)

## 6. Tabla final de resultados

In [ ]:
rows = []
rows.append({'modelo': 'naive', 'val': float('nan'),
             'test': naive_mae, 'dir': float('nan')})
for tipo, res in results_screen.items():
    rows.append({'modelo': f'{tipo} (cribado {EPOCHS_SCREEN}ep)',
                 'val': res['val'], 'test': res['test'], 'dir': res['dir']})
for tipo, res in results_full.items():
    rows.append({'modelo': f'{tipo} (full {EPOCHS_FULL}ep)',
                 'val': res['val'], 'test': res['test'], 'dir': res['dir']})
rows.append({'modelo': f'ENSEMBLE {N_SEEDS}seeds ({TIPO_ENSEMBLE})',
             'val': ens['mae_val'], 'test': ens['mae_test'], 'dir': float('nan')})

df_final = pd.DataFrame(rows).set_index('modelo')
df_final['vs_naive'] = (df_final['test'] - naive_mae) / naive_mae
df_final['test']     = df_final['test'].map('{:.4f}'.format)
df_final['val']      = df_final['val'].apply(lambda x: f'{x:.4f}' if x == x else '—')
df_final['dir']      = df_final['dir'].apply(lambda x: f'{x:.2%}' if x == x else '—')
df_final['vs_naive'] = df_final['vs_naive'].apply(
    lambda x: f'{x:+.1%}' if x == x else '—')
display(df_final)

print(f'\n--- COPIAR ESTOS NUMEROS AL GRUPO ---')
print(f'Naive:    {naive_mae:.4f}')
print(f'Ganador:  {results_full[mejor_tipo]["test"]:.4f}  ({mejor_tipo}, {EPOCHS_FULL}ep)')
print(f'Ensemble: {ens["mae_test"]:.4f}  ({N_SEEDS} seeds)')
print(f'Config:   V_IN={V_IN}  V_OUT={V_OUT}  FFD={FFD_D}  LR={LR_MAP[mejor_tipo]}')

## 7. Exportar modelo y resultados — para el ensemble final

Guarda el modelo ganador (mejor arquitectura individual, pesos del mejor epoch)  
y un JSON con la config. `COMPETICION.ipynb` carga estos ficheros para construir  
el ensemble del equipo.

In [ ]:
NOMBRE = 'dani'   # <- nombre del dueno de este notebook

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)

model_path = f'models/{NOMBRE}.keras'
mejor_modelo.save(model_path)
print(f'Modelo guardado: {model_path}')

meta = {
    'nombre':            NOMBRE,
    'modelo':            mejor_tipo,
    'lr':                LR_MAP[mejor_tipo],
    'V_IN':              int(V_IN),
    'V_OUT':             int(V_OUT),
    'FFD_D':             FFD_D,
    'epochs':            EPOCHS_FULL,
    'n_seeds':           N_SEEDS,
    'mae_val':           results_full[mejor_tipo]['val'],
    'mae_test':          results_full[mejor_tipo]['test'],
    'dir_acc':           results_full[mejor_tipo]['dir'],
    'params':            results_full[mejor_tipo]['params'],
    'mae_ensemble_test': ens['mae_test'],
    'mae_ensemble_val':  ens['mae_val'],
}
json_path = f'results/{NOMBRE}.json'
with open(json_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Resultados guardados: {json_path}')
print(f'\nResumen para el equipo:')
print(f'  Modelo: {mejor_tipo}  |  val={meta["mae_val"]:.4f}  '
      f'test={meta["mae_test"]:.4f}  dir_acc={meta["dir_acc"]:.2%}')
print(f'  Ensemble ({N_SEEDS} seeds): test={ens["mae_test"]:.4f}')

## 8. Experimentos adicionales — probar más modelos y decidir ganador

Sección libre. Prueba aquí lo que quieras más allá del flujo estándar:
arquitecturas distintas, `units` distintos, `dropout`, `V_IN` diferente,
preprocesado alternativo, LR distintos, etc.

**Al terminar**, rellena la tabla y compártela con el equipo para decidir
qué modelo exportar en la sección 7 (criterio: `mae_val`, nunca `mae_test`).

| Experimento | Config | MAE val | MAE test | dir_acc | Notas |
|-------------|--------|---------|----------|---------|-------|
| A | | | | | |
| B | | | | | |
| C | | | | | |

### 💡 Próximos pasos recomendados — más allá del flujo estándar

Las secciones 3–5 ya cubren las **4 arquitecturas custom** (dense / lstm / cnn1d / cnn_lstm).
Para los experimentos adicionales de esta sección considera estas familias:

**Transformers especializados** (ver `docs/transformers_y_huggingface.md`):
- **PatchTST** — divide la serie en parches, mejor que LSTM en series largas
- **iTransformer** — atención entre activos (multivariado); captura correlaciones cruzadas

**Fundacionales** (ver `docs/modelos_fundacionales.md`):
- **Chronos-2** — zero-shot o fine-tuning, multivariado, el más completo
- **TimesFM** — univariado, más simple de configurar
- **Kronos / FinCast** — entrenados específicamente con datos financieros

**Búsqueda en HuggingFace Hub**:
```
huggingface.co/models?pipeline_tag=time-series-forecasting&sort=downloads
```
Filtrar por: contexto que soporta tu V_IN · horizonte que cubre tu V_OUT · multivariado si predices varios activos

**Funcional multi-rama** (ver `docs/modelos_fundacionales.md` §6):
- Útil si el enunciado da precios históricos + indicadores escalares (PMI, RSI, etc.) simultáneamente

> Elige solo 1-2 experimentos realistas para las 4 horas disponibles.

### Experimento A

In [ ]:
# Experimento libre — añadir celdas según necesidad


### Experimento B

### Experimento C

### Decisión final — ¿qué modelo exportar?

Basándote en `mae_val` de la tabla de arriba, decide aquí el ganador.
Si el mejor no es el elegido automáticamente en la sección 4, sobreescribe
antes de ejecutar la sección 7:

```python
# mejor_tipo   = '...'
# mejor_modelo = models_full['...']   # o el modelo de tu experimento
```